# 📈 Task 4 — Data Visualisation Dashboard
**DecodeLabs Data Analytics Internship | Batch 2026**  
**Intern:** Umm-e-Abiha

---

## 🎯 Objective
Build a comprehensive, publication-quality visualisation dashboard that communicates the key business insights derived from the E-Commerce Orders dataset.

## 📋 Deliverables
- Monthly revenue trend (line chart)
- Quantity vs Total Price scatter plot
- Top 5 customers by revenue
- Revenue by coupon code
- Unit price distribution by product (violin plot)
- Order status breakdown by product (grouped bar chart)

## 🧠 Skills Applied
`matplotlib` · `seaborn` · `data storytelling` · `dashboard design`

---

## ⚙️ Environment Setup, Data Load & Clean

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

# ── Global Plot Style ────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor'  : '#1a1a2e',
    'axes.edgecolor'  : '#00d4ff',
    'axes.labelcolor' : '#ffffff',
    'xtick.color'     : '#cccccc',
    'ytick.color'     : '#cccccc',
    'text.color'      : '#ffffff',
    'grid.color'      : '#2a2a4a',
    'grid.linestyle'  : '--',
    'grid.alpha'      : 0.4,
})

CYAN   = '#00d4ff'
ORANGE = '#ff6b35'
GREEN  = '#39ff14'
PURPLE = '#bf5af2'
YELLOW = '#ffd60a'
PINK   = '#ff2d78'
COLORS = [CYAN, ORANGE, GREEN, PURPLE, YELLOW, PINK, '#00ff88']

print("✅ All libraries imported successfully!")

df_raw = pd.read_excel('Dataset_for_Data_Analytics.xlsx')
df = df_raw.copy()
df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')
df = df.drop_duplicates()
df['Date']      = pd.to_datetime(df['Date'])
df['Year']      = df['Date'].dt.year
df['Month']     = df['Date'].dt.month
df['MonthName'] = df['Date'].dt.strftime('%b')
for col in ['Product', 'OrderStatus', 'PaymentMethod', 'ReferralSource']:
    df[col] = df[col].str.strip().str.title()
print(f"Clean dataset ready: {df.shape[0]:,} rows × {df.shape[1]} columns")


## 4.1 Full Analytics Dashboard (6-Panel)

In [ ]:
# ── Task 4: Full 6-Panel Visualisation Dashboard ─────────────────────────────
fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('E-Commerce Analytics Dashboard\nDecodeLabs Internship 2026 — Umm-e-Abiha',
             color=CYAN, fontsize=18, fontweight='bold', y=1.01)

gs = fig.add_gridspec(3, 3, hspace=0.48, wspace=0.38)

# ── Chart 1: Monthly Revenue Trend ───────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
monthly = (df.groupby(['Year', 'Month'])['TotalPrice'].sum().reset_index()
             .assign(Period=lambda x: x['Year'].astype(str) + '-' + x['Month'].astype(str).str.zfill(2))
             .sort_values(['Year', 'Month']))
x_idx = range(len(monthly))
ax1.plot(x_idx, monthly['TotalPrice'], color=CYAN, linewidth=2.5,
         marker='o', markersize=5, markerfacecolor=ORANGE)
ax1.fill_between(x_idx, monthly['TotalPrice'], alpha=0.15, color=CYAN)
ax1.set_xticks(list(x_idx))
ax1.set_xticklabels(monthly['Period'], rotation=45, ha='right', fontsize=7)
ax1.set_title('Monthly Revenue Trend (Jan 2023 – Jun 2025)', color=CYAN, fontsize=13)
ax1.set_ylabel('Revenue ($)')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))

# ── Chart 2: Quantity vs TotalPrice Scatter ───────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
for i, prod in enumerate(df['Product'].unique()):
    sub = df[df['Product'] == prod]
    ax2.scatter(sub['Quantity'], sub['TotalPrice'], color=COLORS[i],
                alpha=0.5, s=15, label=prod)
ax2.set_title('Quantity vs Total Price', color=CYAN, fontsize=12)
ax2.set_xlabel('Quantity'); ax2.set_ylabel('Total Price ($)')
ax2.legend(fontsize=6, loc='upper left', framealpha=0.3)

# ── Chart 3: Top 5 Customers by Revenue ──────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
top5 = df.groupby('CustomerID')['TotalPrice'].sum().nlargest(5)
ax3.barh(top5.index, top5.values, color=COLORS[:5], edgecolor='#0d0d0d')
ax3.set_title('Top 5 Customers by Revenue', color=CYAN, fontsize=11)
ax3.set_xlabel('Revenue ($)')
for i, v in enumerate(top5.values):
    ax3.text(v + 10, i, f'${v:,.0f}', va='center', fontsize=8, color='white')

# ── Chart 4: Revenue by Coupon Code ──────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
coupon_rev = df.groupby('CouponCode')['TotalPrice'].sum().sort_values(ascending=False).head(6)
ax4.bar(coupon_rev.index, coupon_rev.values, color=COLORS[:6], edgecolor='#0d0d0d')
ax4.set_title('Revenue by Coupon Code (Top 6)', color=CYAN, fontsize=11)
ax4.set_ylabel('Revenue ($)'); ax4.tick_params(axis='x', rotation=20)
for i, v in enumerate(coupon_rev.values):
    ax4.text(i, v + 100, f'${v:,.0f}', ha='center', fontsize=7, color='white')

# ── Chart 5: Unit Price Distribution — Violin ─────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
prod_list = df['Product'].unique()
parts = ax5.violinplot([df[df['Product']==p]['UnitPrice'].values for p in prod_list],
                        showmedians=True)
for pc, color in zip(parts['bodies'], COLORS):
    pc.set_facecolor(color); pc.set_alpha(0.6)
parts['cmedians'].set_color(ORANGE); parts['cmedians'].set_linewidth(2)
ax5.set_xticks(range(1, len(prod_list)+1))
ax5.set_xticklabels(prod_list, rotation=30, ha='right', fontsize=8)
ax5.set_title('Unit Price Distribution by Product', color=CYAN, fontsize=11)
ax5.set_ylabel('Unit Price ($)')

# ── Chart 6: Order Status by Product — Grouped Bar ───────────────────────────
ax6 = fig.add_subplot(gs[2, :])
pivot = df.pivot_table(index='Product', columns='OrderStatus',
                        values='OrderID', aggfunc='count', fill_value=0)
pivot.plot(kind='bar', ax=ax6, color=COLORS[:pivot.shape[1]],
           edgecolor='#0d0d0d', linewidth=0.8, width=0.75)
ax6.set_title('Order Status Breakdown by Product', color=CYAN, fontsize=13)
ax6.set_ylabel('Number of Orders'); ax6.set_xlabel('')
ax6.tick_params(axis='x', rotation=25)
ax6.legend(title='Order Status', fontsize=9, title_fontsize=10,
           framealpha=0.3, loc='upper right')

plt.savefig('task4_visualization.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print("✅ Dashboard saved → task4_visualization.png")


## 💡 Task 4 — Chart-by-Chart Insights

| Chart | Type | Key Insight |
|-------|------|-------------|
| Monthly Revenue Trend | Line | Revenue is stable with monthly fluctuations; no strong seasonality detected |
| Qty vs TotalPrice | Scatter | Higher unit price correlates with higher bill; products segment clearly |
| Top 5 Customers | Horizontal Bar | A small customer cohort drives disproportionate revenue (Pareto effect) |
| Revenue by Coupon | Bar | `NO_COUPON` customers generate the most revenue; discount dependency is low |
| Unit Price Violin | Violin | Chair, Laptop, Tablet show the widest price ranges (broad market coverage) |
| Status by Product | Grouped Bar | Return and cancellation rates are consistent across all product types |

> **Visualisation Principle Applied:** Each chart type was chosen to best represent the data structure — line for trends, scatter for relationships, violin for distributions, grouped bar for multi-category comparisons.
